# Rung-1 comparison: 1a

Tasks: fastapi_15588,fastapi_15589,rich_3882,rich_3470

Generated by scripts/make_kaggle_notebook.py. Attach competition data, the wheelhouse dataset and the 31B model; accelerator L4 ×4; Run All; stop the session when done.

In [ ]:
# 1. Install the competition wheelhouse BEFORE anything imports google.adk
#    (Kaggle preinstalls another google-adk; importing it first causes pydantic class clashes).
import glob, importlib, os, shutil, subprocess, sys
from pathlib import Path

os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
if not WHEELHOUSE_DIR.exists():
    WHEELHOUSE_DIR = Path(glob.glob('/kaggle/input/**/gemma-4-developer-agent-wheelhouse', recursive=True)[0])

for pth in glob.glob('/usr/local/lib/python*/*-packages/*cutlass*.pth'):
    try:
        os.unlink(pth)
    except OSError:
        pass

tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target = tmp_whl / (w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name)
    if not target.exists():
        os.symlink(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
assert wheels, f'No wheels found in {WHEELHOUSE_DIR}'
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
importlib.invalidate_caches()
print('Wheelhouse installation complete.')

In [ ]:
# 2. Write the submissions to evaluate
import re
DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
SUB_ROOT = WORKING_DIR / 'submissions'
TASK_IDS = ['fastapi_15588', 'fastapi_15589', 'rich_3882', 'rich_3470']
CONFIGS = ['1a']
REPEATS = 2
EMBEDDED = {
 "1a": {
  "agent.yaml": "name: executor\ndescription: Single agent that localizes, edits, verifies and submits a fix on its own.\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  # Declared only so a call never crashes the task: the harness task message may advertise them,\n  # and an undeclared tool call ends the task and discards the patch (forum 745028).\n  - search_similar_code\n  - get_code_neighbors\n  - get_code_subgraph\ngenerate_content_config: !include configs/sampling.yaml\n",
  "configs/sampling.yaml": "# Thinking ON since 10-09 (rung-1l): with \"thinking off\" the model reasoned anyway, invisibly, at the same cost\n# (2.5-3.2x hidden tokens since wheelhouse 0.2.13); thinking on makes the thoughts visible and keeps them.\n# Budget 1,024: median 119, p90 677 thinking+answer tokens per step. Sampling per the Gemma 4 model card.\ntemperature: 1.0\ntop_p: 0.95\ntop_k: 64\nmax_output_tokens: 2048\nthinking_config:\n  include_thoughts: true\n  thinking_budget: 1024\n",
  "eval_config.yaml": "evaluation:\n  timeout_seconds: 240\n  max_tool_calls: 60\n  max_time_minutes: 5.0\n  max_turns: 90\n",
  "prompts/system.md": "You are an autonomous software engineer fixing one task in the repository at /workspace.\nNobody will answer questions. Never ask; decide from the task text, the code and the tests.\n\n## The task (verbatim)\n{problem_description}\n\n## Budget\nYou have about 5 minutes, and time is spent on the text you write.\n- Every reply: one short plain-text progress line, then exactly one tool call. No long analysis.\n- Target: locate within ~8 calls, first edit by ~call 15, submit by ~call 30.\n- get_status is free: check it when unsure. Under 75 seconds left: make your best edit and submit.\n\n## Workflow\n1. Find the source package (the workspace listing you were given often hides it):\n   run_command: git ls-files '*.py' | grep -v -e tests -e docs | head -30\n2. Locate: run_command: git grep -n 'NAME' -- '*.py' | head -20\n   Short or title-only task: git log --oneline -15, git log -S'SYMBOL' --oneline | head,\n   git show SHA --stat, and look at docs/, docs_src/ and similar existing features.\n   Prefer git grep; use the graph tools only with an exact function or class name.\n3. Read line ranges with run_command: cat -n path/to/file.py | sed -n '40,90p'\n   At most 60 lines at a time. Do not use read_file.\n4. Baseline: run the nearest test file once and note which tests already fail:\n   run_command: python -m pytest tests/test_x.py -q -rf 2>&1 | tail -15\n   Reproduce if quick: write /tmp/repro.py at most twice in the whole task and re-run it by name. A repro that\n   does not show the bug is not a reason to skip the change: every task needs a source change, so go by the task\n   text.\n5. Edit: small edit_file changes in source files. Never edit tests, conftest.py, pytest.ini or config.\n   Copy old_string exactly from what sed printed, and keep it short: 1 to 5 consecutive lines.\n   If edit_file returns an error, never repeat the same call. Shorten old_string, or edit with run_command:\n   python - << 'EOF'\n   p = 'pkg/module.py'\n   s = open(p).read()\n   old = '''exact old lines'''\n   new = '''new lines'''\n   assert s.count(old) == 1\n   open(p, 'w').write(s.replace(old, new))\n   EOF\n   Match the conventions already used in the same file: error message wording and format, naming, validator and\n   helper style. Cover all equivalent cases (for example both \\r and \\n when the task is about line breaks).\n6. Verify: re-run the repro and the same test file as in the baseline:\n   run_command: python -m pytest tests/test_x.py -q -rf 2>&1 | tail -15\n   A test that passed in the baseline and fails now means the change is wrong: adjust the change. Never undo your\n   change completely.\n7. Pre-submit: run_command: git status --short && git diff --stat\n   Only the source files you meant to change may appear. Delete anything else you created in /workspace with rm.\n   An empty diff always scores zero: if nothing is changed, make your best edit before submitting.\n8. Call submit_patch, then reply with one short sentence. Make no edits after submitting.\n\n## Files: where scratch work goes\n- Scratch files (repro scripts, logs) live in /tmp and are created ONLY with run_command and a heredoc:\n  cat > /tmp/name.py << 'EOF' ... EOF\n- write_file and edit_file work only inside /workspace: use them only for the real source changes.\n- Every file created in /workspace ends up in the graded patch. Never create repro or test scripts there.\n\n## Tool calls\n- Call tools only through the tool-calling interface, never by writing a tool call as text.\n- The tool name is always one of your tools (run_command, edit_file, ...). Shell commands such as\n  git, grep, python or cat go inside run_command, never as the tool name.\n- Never use backticks: write file paths, names and code as plain text. Backticks break tool arguments.\n- Shell quoting: use single quotes inside commands (git grep -n 'class Foo'). Never write backslash-escaped quotes.\n\n## Rules\n- Keep outputs small: pipe long commands through | head -40 or | tail -20; never print whole files.\n  rg and tree are not installed; use git grep. No pip, no network.\n- Never run the identical command twice. If it did not help, change it or move on.\n- Git is read-only: log, show, grep, blame, diff, status. Never checkout, restore, reset, stash, clean,\n  rebase or commit.\n- Ambiguous task: choose the reading that matches the exact wording and existing code conventions,\n  and keep to it.\n- An imperfect edit beats no edit: the working tree is graded even if you run out of time.\n"
 }
}

shutil.rmtree(SUB_ROOT, ignore_errors=True)
for name, files in EMBEDDED.items():
    for rel, text in files.items():
        path = SUB_ROOT / name / rel
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text, encoding='utf-8')

if 'sample' in CONFIGS:
    # Google's sample as shipped, minus its LoRA adapters.
    sample = SUB_ROOT / 'sample'
    shutil.copytree(DATA_DIR / 'sample_submission', sample)
    shutil.rmtree(sample / 'adapters', ignore_errors=True)
    for yml in list(sample.rglob('*.yaml')):
        yml.write_text(re.sub(r'^adapter:.*\n', '', yml.read_text(), flags=re.M))
    (sample / 'eval_config.yaml').write_text(
        'evaluation:\n  timeout_seconds: 240\n  max_tool_calls: 100\n  max_time_minutes: 5.0\n  max_turns: 90\n')

for name in CONFIGS:
    print(name, sorted(p.relative_to(SUB_ROOT / name).as_posix() for p in (SUB_ROOT / name).rglob('*') if p.is_file()))

In [ ]:
# 3. Start vLLM once for all configs (no adapters → LoRA off → more KV cache). Takes ~6–8 min.
import litellm, torch
from adk_submission import VllmConfig, VllmServer

litellm.drop_params = True
TARGET_MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
MODEL_PATH = Path(glob.glob('/kaggle/input/models/google/gemma-4/**/gemma-4-31b-it-qat-w4a16-ct/2', recursive=True)[0])
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=False,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
)
server_instance = VllmServer(vllm_cfg)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')
models = server_instance.create_model_registry(aliases=[TARGET_MODEL_NAME], model_prefix='openai/', api_key='EMPTY')

# Installed harness versions (the wheelhouse changes over time; rung-1i notes).
from importlib.metadata import version, PackageNotFoundError
def _ver(p):
    try:
        return version(p)
    except PackageNotFoundError:
        return None
print('VERSIONS', {p: _ver(p) for p in ('adk-submission', 'swegemma', 'adk-eval-core', 'google-adk', 'vllm')})

# Sampling diagnostic (rung-1j): replay the REAL first orchestrator request (full 1b prompt + harness task message)
# under several sampling settings, thinking off. Counts hidden reasoning, thought markers leaking into text, and
# whether a proper tool call came back. Task text is read at runtime from the competition data (never committed).
def _sampling_diagnostic(task_ids=("rich_3882", "fastapi_15589"), samples=3):
    import json as _json, time, types, urllib.request as _url
    from swegemma.harness.agent_runner import build_agent_prompt
    _base = server_instance.base_url.rstrip("/")
    _base = _base if _base.endswith("/v1") else _base + "/v1"
    _served = _json.load(_url.urlopen(_base + "/models"))["data"][0]["id"]
    def _fn(name, desc, props=None, req=()):
        return {"type": "function", "function": {"name": name, "description": desc, "parameters":
                {"type": "object", "properties": props or {}, "required": list(req)}}}
    _s = {"type": "string"}; _i = {"type": "integer"}
    _tools = [_fn("run_command", "Run a shell command in the workspace.", {"command": _s}, ["command"]),
              _fn("read_file", "Read a file.", {"filepath": _s, "start_line": _i, "end_line": _i}, ["filepath"]),
              _fn("get_status", "Budget and progress."), _fn("submit_patch", "Submit the current diff."),
              _fn("search_similar_code", "Semantic code search.", {"query": _s}, ["query"]),
              _fn("get_code_neighbors", "Graph neighbors of a symbol.", {"symbol": _s}, ["symbol"]),
              _fn("get_code_subgraph", "Graph around a symbol.", {"symbol": _s}, ["symbol"]),
              _fn("executor", "Makes the code change.", {"request": _s}, ["request"]),
              _fn("verifier", "Checks the change.", {"request": _s}, ["request"])]
    _tasks = {}
    for _line in (DATA_DIR / "tasks.jsonl").read_text().splitlines():
        _d = _json.loads(_line)
        if _d.get("instance_id") in task_ids:
            _tasks[_d["instance_id"]] = types.SimpleNamespace(**_d)
    class _NS(types.SimpleNamespace):
        def __getattr__(self, _name):  # any field the harness reads but we did not set -> None
            return None
    _cfg = _NS(budget=_NS(time_minutes=5.0, tool_calls=80, turns=90), harness=_NS(command_timeout_seconds=240, max_stdout_chars=5000),
               enable_sandbox_testing=True)
    _instr = (SUB_ROOT / "1b" / "prompts" / "orchestrator.md").read_text()
    _settings = [("T1.0 p0.95 k64", 1.0, 0.95, 64), ("T1.0 p0.80 k20", 1.0, 0.80, 20), ("T0.6 p0.95 k64", 0.6, 0.95, 64),
                 ("T0.4 p0.95 k64", 0.4, 0.95, 64), ("T0.2 p0.95 k64", 0.2, 0.95, 64)]
    _summary = {}
    for _tid, _task in _tasks.items():
        try:
            _user = build_agent_prompt(_task, _cfg, "", declared_tools=[t["function"]["name"] for t in _tools])
        except Exception as _e:
            print("build_agent_prompt failed, using plain task text:", repr(_e))
            _user = "Problem Statement:\n" + _task.problem_statement
        _msgs = [{"role": "system", "content": _instr.replace("{problem_description}", _task.problem_statement)},
                 {"role": "user", "content": _user}]
        for _name, _t, _p, _k in _settings:
            for _rep in range(samples):
                _body = {"model": _served, "messages": _msgs, "tools": _tools, "temperature": _t, "top_p": _p,
                         "top_k": _k, "max_tokens": 1024, "chat_template_kwargs": {"enable_thinking": False}}
                _t0 = time.time()
                _req = _url.Request(_base + "/chat/completions", data=_json.dumps(_body).encode(),
                                    headers={"Content-Type": "application/json"})
                _r = _json.load(_url.urlopen(_req, timeout=600))
                _m = _r["choices"][0]["message"]
                _reason = _m.get("reasoning_content") or _m.get("reasoning") or ""
                _content = _m.get("content") or ""
                _ok = bool(_m.get("tool_calls"))
                _leak = "<|channel>" in _content or "<channel|>" in _content
                _row = (time.time() - _t0, _r["usage"]["completion_tokens"], len(_reason), _leak, _ok)
                _summary.setdefault(_name, []).append(_row)
                print(f"{_tid:14} {_name}  #{_rep + 1}: {_row[0]:5.1f}s  out_tok={_row[1]:5}  reasoning_chars={_row[2]:5}  "
                      f"marker_in_text={_leak}  tool_call={_ok}  prompt_tok={_r['usage']['prompt_tokens']}")
                if (_reason or _leak) and _rep == 0:
                    print("    starts:", repr((_reason or _content)[:160]))
    print("SAMPLING SUMMARY (per setting over all tasks and samples)")
    for _name, _rows in _summary.items():
        _n = len(_rows)
        print(f"  {_name}: mean {sum(r[0] for r in _rows) / _n:5.1f}s  mean out_tok {sum(r[1] for r in _rows) / _n:6.0f}  "
              f"with reasoning {sum(r[2] > 0 for r in _rows)}/{_n}  marker leaks {sum(r[3] for r in _rows)}/{_n}  "
              f"proper tool calls {sum(r[4] for r in _rows)}/{_n}")
RUN_SAMPLING_DIAGNOSTIC = False  # rung-1k answered it: direct calls never think
try:
    if RUN_SAMPLING_DIAGNOSTIC:
        _sampling_diagnostic()
except Exception as _e:  # never block the evaluation
    print("sampling diagnostic failed:", repr(_e))

In [ ]:
# 4. Evaluate every config on the same tasks (Evaluator configured like the scorer; see scripts/run_eval.py)
import asyncio, concurrent.futures, time, yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator

DATA = DATA_DIR  # default used by build_config (copied from scripts/run_eval.py)


def read_eval_config(submission_dir):
    """The four eval_config.yaml keys the scorer reads (defaults as in the host notebook)."""
    path = Path(submission_dir) / "eval_config.yaml"
    raw = yaml.safe_load(path.read_text()) if path.exists() else {}
    section = (raw or {}).get("evaluation", raw or {})
    return {
        "timeout_seconds": int(section.get("timeout_seconds", 300)),
        "max_tool_calls": section.get("max_tool_calls"),
        "max_time_minutes": float(section.get("max_time_minutes", 5.0)),
        "max_turns": section.get("max_turns"),
    }


def wheels_dir_for(data_dir):
    """Official test wheels, plus data/wheels_extra if present (local only: test deps missing from the public set,
    forum 745775). Merged as links into data/wheels_merged so the official folder stays untouched."""
    official, extra = Path(data_dir) / "wheels", Path(data_dir) / "wheels_extra"
    if not extra.is_dir():
        return official
    merged = Path(data_dir) / "wheels_merged"
    merged.mkdir(exist_ok=True)
    for wheel in [*official.glob("*.whl"), *extra.glob("*.whl")]:
        link = merged / wheel.name
        if not link.exists():
            link.hardlink_to(wheel)
    return merged


def build_config(submission_dir, task_ids, results_dir, models, data_dir=DATA, sandbox="docker",
                 adapter_manifest=None, display_mode="single"):
    limits, gen_constraints = build_submission_limits()
    budgets = read_eval_config(submission_dir)
    return EvalConfig(
        tasks_path=Path(data_dir) / "tasks.jsonl",
        snapshots_dir=Path(data_dir) / "snapshots",
        results_dir=Path(results_dir),
        submission_dir=Path(submission_dir),
        models=models,
        sandbox=sandbox,
        wheels_dir=wheels_dir_for(data_dir),
        graph_dir=str(Path(data_dir) / "graphs"),
        embeddings_dir=str(Path(data_dir) / "embeddings"),
        task_ids=list(task_ids),
        limits=limits,
        generation_constraints=gen_constraints,
        adapter_manifest=adapter_manifest,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(
            compaction_interval=5, overlap_size=2, token_threshold=14336, event_retention_size=5,
        ),
        display_mode=display_mode,
        **budgets,
    )


def run_sync(coro):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(coro)).result()
    return asyncio.run(coro)


RESULTS = WORKING_DIR / 'results'
shutil.rmtree(RESULTS, ignore_errors=True)
# Repeats are interleaved (r1 of every config, then r2, ...) so a session that dies early still has complete pairs.
RUNS = [(name, f'{name}_r{rep + 1}' if REPEATS > 1 else name) for rep in range(REPEATS) for name in CONFIGS]
wall = {}
for name, key in RUNS:
    start = time.time()
    config = build_config(SUB_ROOT / name, TASK_IDS, RESULTS / key, models, data_dir=DATA_DIR,
                          sandbox='subprocess', display_mode='single')
    result = run_sync(Evaluator(config).run())
    wall[key] = time.time() - start
    print(f'==> {key}: {result.resolved}/{result.total} resolved in {wall[key] / 60:.1f} min')
    shutil.make_archive(str(WORKING_DIR / 'results'), 'zip', RESULTS)  # partial results survive a crash

In [ ]:
# 5. Comparison table + results.zip (download it from the Output panel), then STOP THE SESSION.
import json
from collections import Counter


NUDGE_PREFIXES = ("Please continue your work", "Your previous response reached the token limit")


HELPER_FORMATS = {"reader": "LOCATION:", "executor": "CHANGED:", "verifier": "VERDICT:"}


TEST_PATHS = ("tests/", "test_", "conftest.py", "pytest.ini")


def bucket(task, patch=""):
    """Coarse failure bucket from the harness result (patch text comes from patches/<id>.patch)."""
    error = (task.get("error") or "").lower()
    patch = patch or task.get("agent_patch") or ""
    if task.get("resolved"):
        return "resolved"
    if "context" in error and ("window" in error or "length" in error):
        return "overflow"
    if "not found" in error and "tool" in error:
        return "unknown tool"
    if not patch.strip():
        return "timeout, no patch" if "timeout" in error or "timed out" in error else "no patch"
    changed = [line.split(" b/")[-1] for line in patch.splitlines() if line.startswith("diff --git")]
    if any(any(t in f for t in TEST_PATHS) for f in changed):
        return "edited tests"
    return "wrong fix"


def trace_stats(trace):
    steps = trace.get("steps", [])
    stats = Counter()
    tools = Counter()
    commands = Counter()
    helper_ok = Counter()
    helper_total = Counter()
    last_prompt = {}  # per agent: helpers have their own conversations
    peak_prompt = 0
    for step in steps:
        message = str(step.get("message") or "")
        if step.get("source") != "agent" and message.startswith(NUDGE_PREFIXES):
            stats["nudges"] += 1
        metrics = step.get("metrics") or {}
        prompt = metrics.get("prompt_tokens")
        if prompt:
            stats["model_calls"] += 1
            stats["output_tokens"] += metrics.get("completion_tokens") or 0
            peak_prompt = max(peak_prompt, prompt)
            # Compaction replaces old events with a summary: the agent's next prompt is much smaller than its last.
            # A helper (agent_tool) starts a fresh conversation on every call, so a new call resets its baseline.
            author = (step.get("extra") or {}).get("author")
            previous = last_prompt.get(author)
            if previous and prompt < 0.7 * previous:
                stats["compactions"] += 1
            last_prompt[author] = prompt
        for call in step.get("tool_calls") or []:
            name = call.get("function_name")
            tools[name] += 1
            if name == "run_command":
                commands[(call.get("arguments") or {}).get("command", "")] += 1
            if name in HELPER_FORMATS:
                last_prompt.pop(name, None)
        # Observations are tagged with the tool that produced them. Helper (agent_tool) replies arrive as
        # {"raw": "<final text>"} in the step whose observation is tagged with the helper's name.
        observation = step.get("observation") or {}
        source = (observation.get("extra") or {}).get("tool_name")
        content = str(observation.get("content", ""))
        if '"status": "error"' in content[:40] or "'status': 'error'" in content[:40] or content.startswith('{"error"'):
            stats["tool_errors"] += 1
        if source in HELPER_FORMATS:
            helper_total[source] += 1
            try:
                reply = json.loads(content).get("raw", content)
            except (ValueError, AttributeError):
                reply = content
            if HELPER_FORMATS[source] in reply[:300]:
                helper_ok[source] += 1
            if "<|tool_call>" in reply:
                stats["broken_calls"] += 1
        if "<|tool_call>" in message:
            stats["broken_calls"] += 1
    stats["peak_prompt"] = peak_prompt
    stats["repeated_cmds"] = sum(n - 1 for n in commands.values() if n > 1)
    helpers = {h: f"{helper_ok[h]}/{helper_total[h]}" for h in helper_total}
    return stats, tools, helpers


def analyze(results_dir):
    results_dir = Path(results_dir)
    tasks = [json.loads(line) for line in (results_dir / "task_results.jsonl").read_text().splitlines() if line]
    print(f"\n## {results_dir.name}: {sum(t.get('resolved', False) for t in tasks)}/{len(tasks)} resolved")
    header = (f"{'task':<16} {'outcome':<18} {'sec':>5} {'calls':>5} {'out_tok':>7} {'peak_in':>7} "
              f"{'compact':>7} {'nudge':>5} {'repeat':>6} {'errors':>6} {'broken':>6}  tools / helpers")
    print(header)
    print("-" * len(header))
    for task in tasks:
        task_id = task.get("instance_id") or task.get("task_id")
        trace_file = results_dir / "traces" / f"trace_{task_id}.json"
        trace = json.loads(trace_file.read_text()) if trace_file.exists() else {}
        stats, tools, helpers = trace_stats(trace)
        patch_file = results_dir / "patches" / f"{task_id}.patch"
        patch = patch_file.read_text() if patch_file.exists() else ""
        tool_list = ", ".join(f"{name}×{n}" for name, n in tools.most_common())
        helper_list = (" | format ok " + ", ".join(f"{h} {v}" for h, v in helpers.items())) if helpers else ""
        print(f"{task_id:<16} {bucket(task, patch):<18} {task.get('duration_seconds', 0):>5.0f} {stats['model_calls']:>5} "
              f"{stats['output_tokens']:>7} {stats['peak_prompt']:>7} {stats['compactions']:>7} "
              f"{stats['nudges']:>5} {stats['repeated_cmds']:>6} {stats['tool_errors']:>6} {stats['broken_calls']:>6}  "
              f"{tool_list}{helper_list}")
        if patch:
            changed = [line.split(" b/")[-1] for line in patch.splitlines() if line.startswith("diff --git")]
            print(f"{'':<16} patch: {', '.join(changed)}")
        if task.get("error"):
            print(f"{'':<16} error: {task['error'][:150]}")


for _, key in RUNS:
    analyze(RESULTS / key)
shutil.make_archive(str(WORKING_DIR / 'results'), 'zip', RESULTS)
print('\nWall clock per config (min):', {k: round(v / 60, 1) for k, v in wall.items()})
print('Wrote /kaggle/working/results.zip')